## Tests — Recommendations + Feedback Loop

Validates deterministic recommendation rules and feedback filtering in `model/recommend_core.py`.

Covers:
- recurring merchant detection
- top discretionary category + top merchant cutback detection
- recommendation generation + dismiss feedback filtering


## Imports

In [6]:
from __future__ import annotations

import sys
import tempfile
from datetime import date
from pathlib import Path

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "artifacts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/ and artifacts/")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.recommend_core import (
    detect_recurring_merchants,
    detect_top_category_merchant,
    generate_recommendations,
    load_feedback,
    write_feedback,
)


## Synthetic transactions

In [7]:
client_id = 1696
as_of = date(2011, 6, 15)

# Recurring subscription-like merchant across 4 months
tx = pd.DataFrame(
    {
        "transaction_dt": [
            "2011-03-02",
            "2011-04-02",
            "2011-05-02",
            "2011-06-02",
        ],
        "amount_usd": [12.99, 12.99, 12.99, 12.99],
        "merchant_id": ["MERCH_SUB"] * 4,
        "category": ["Subscriptions"] * 4,
        "mcc_description": ["Subscription"] * 4,
        "is_discretionary": [True] * 4,
        "merchant_city": ["ONLINE"] * 4,
        "merchant_state": [None] * 4,
    }
)

# Dining: MERCH_DINE_A is the top merchant inside the top discretionary category
tx2 = pd.DataFrame(
    {
        "transaction_dt": ["2011-04-10", "2011-05-10", "2011-06-10", "2011-06-12"],
        "amount_usd": [80.0, 90.0, 100.0, 40.0],
        "merchant_id": ["MERCH_DINE_A", "MERCH_DINE_A", "MERCH_DINE_A", "MERCH_DINE_B"],
        "category": ["Dining"] * 4,
        "mcc_description": ["Eating Places"] * 4,
        "is_discretionary": [True] * 4,
        "merchant_city": ["Dallas", "Dallas", "Dallas", "Austin"],
        "merchant_state": ["TX"] * 4,
    }
)

tx = pd.concat([tx, tx2], ignore_index=True)


## Detect recurring merchants

In [8]:
rec = detect_recurring_merchants(tx, as_of_date=as_of)
assert not rec.empty
assert (rec["merchant_id"] == "MERCH_SUB").any()


## Detect top category + top merchant cutback

In [9]:
hit = detect_top_category_merchant(tx, as_of_date=as_of, lookback_days=90)
assert hit is not None
assert hit["category"] == "Dining"
assert hit["merchant_id"] == "MERCH_DINE_A"
assert float(hit["merchant_spend_usd"]) == 270.0
assert float(hit["category_spend_usd"]) == 310.0
assert float(hit["estimated_monthly_savings_usd"]) > 0


## Generate up to 3 recommendations + feedback filtering

In [10]:
with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    (root / "artifacts").mkdir(parents=True, exist_ok=True)
    recs = generate_recommendations(
        tx,
        client_id=client_id,
        as_of_date=as_of,
        monthly_limit_usd=200.0,
        root=root,
        max_recommendations=3,
    )
    assert 1 <= len(recs) <= 3
    rules = {r.rule for r in recs}
    assert "cutback_top_category_merchant" in rules or "recurring_subscription" in rules

    cutbacks = [r for r in recs if r.rule == "cutback_top_category_merchant"]
    if cutbacks:
        assert cutbacks[0].meta["merchant_id"] == "MERCH_DINE_A"
        assert cutbacks[0].meta["category"] == "Dining"

    # dismiss first recommendation and confirm it no longer appears
    first_id = recs[0].rec_id
    write_feedback(root, client_id=client_id, rec_id=first_id, status="dismissed")
    recs2 = generate_recommendations(
        tx,
        client_id=client_id,
        as_of_date=as_of,
        monthly_limit_usd=200.0,
        root=root,
        max_recommendations=3,
    )
    assert all(r.rec_id != first_id for r in recs2)
    assert load_feedback(root, client_id=client_id)[first_id]["status"] == "dismissed"

print("Recommend tests: PASS")


Recommend tests: PASS
